# Session 1 — Data Collection with HTTP and APIs

**Data Science & Machine Learning: from Skills to the Project**  
**Duration:** 90 minutes  
**Case study:** collect a small catalogue of books from local sample responses.

We will build a repeatable, documented dataset without relying on a network connection. A public, no-key API demonstration is available as an optional step.

## Prerequisites, outcomes and agenda

You should be comfortable with Python functions, lists and dictionaries, and with Git. The notebook uses **pandas**, **requests** and the Python standard library.

By the end, you can:
- choose a suitable data source for a question;
- retrieve data through HTTP/REST, and interpret status codes, parameters, timeouts and authentication;
- parse JSON, CSV and XML into structured records;
- validate and persist a consistent dataset for later analysis.

| Minutes | Activity |
|---:|---|
| 0–10 | Sources and selection |
| 10–30 | HTTP, REST, parameters and errors |
| 30–45 | JSON and nested records |
| 45–60 | CSV, XML and format comparison |
| 60–75 | Guided catalogue exercise |
| 75–90 | Independent task and reflection |

Run cells from top to bottom. The local examples are deterministic; no credentials or external datasets are needed.

## 1. Choose a data collection approach

Common options include downloading a file, querying an API, collecting from a database, or scraping a web page. Prefer an official, documented API or downloadable dataset when it provides the fields and licence you need. Scrape only when permitted, and check the site's terms, robots guidance and request limits.

Choose against the **question**, required fields, freshness, reliability, access conditions and expected effort—not simply the format that looks easiest. For our small book catalogue, a local CSV is sufficient for reproducible practice; an API is useful for practising retrieval and pagination.

**Interpretation:** the local snapshot is right for repeatable parsing practice. A live API would suit a task requiring current catalogue records; scraping is not the default when a structured source exists.

In [ ]:
from pathlib import Path
import csv
import json
import xml.etree.ElementTree as ET

import pandas as pd
import requests

source_options = {
    "local catalogue CSV": {"offline": True, "structured": True, "freshness": "snapshot"},
    "public books API": {"offline": False, "structured": True, "freshness": "current"},
    "web-page scraping": {"offline": False, "structured": False, "freshness": "current"},
}
chosen_source = "local catalogue CSV"
assert chosen_source in source_options
assert source_options[chosen_source]["offline"]
print(f"Chosen for this lesson: {chosen_source} (reproducible offline)")

## 2. HTTP requests, REST, status codes and errors

HTTP sends a request to a URL and receives a response. A REST API commonly exposes resources through URLs and uses methods such as `GET` to read them. Query parameters refine a request (for example, a title and a result limit); headers can carry content preferences or authentication.

A `2xx` status indicates success, `4xx` usually points to a request or access problem, and `5xx` indicates a server-side problem. Set a timeout, check the status before parsing, and handle timeout, connection, HTTP and decoding errors explicitly. Never assume that a successful HTTP response contains valid JSON.

Authentication is often an API key or bearer token in a header. Keep real secrets outside notebooks (for example, in a protected environment variable); do not commit or print them. This lesson needs no credentials.

**Interpretation:** the prepared URL demonstrates safe parameter encoding. The helper sets a finite timeout, checks HTTP status and surfaces common failures; a bearer-token header would look like `{"Authorization": "Bearer <token>"}`, where the placeholder is not a credential. Open Library's public search endpoint requires no key, but depends on network availability and may rate-limit responses.

In [ ]:
from urllib.parse import parse_qs, urlparse

prepared = requests.Request(
    "GET",
    "https://openlibrary.org/search.json",
    params={"title": "The Hobbit", "limit": 3},
).prepare()
query = parse_qs(urlparse(prepared.url).query)
assert query == {"title": ["The Hobbit"], "limit": ["3"]}
print("Prepared URL:", prepared.url)

def get_json(url, params=None, timeout=10):
    """Fetch JSON and raise a clear error instead of hiding request failures."""
    try:
        response = requests.get(url, params=params, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except requests.Timeout as exc:
        raise RuntimeError(f"Request timed out after {timeout} seconds") from exc
    except requests.ConnectionError as exc:
        raise RuntimeError("Could not connect to the data source") from exc
    except requests.HTTPError as exc:
        status = exc.response.status_code if exc.response is not None else "unknown"
        raise RuntimeError(f"The server returned HTTP {status}") from exc
    except requests.exceptions.JSONDecodeError as exc:
        raise ValueError("The response body was not valid JSON") from exc

status_examples = {200: "success", 404: "client/request issue", 429: "rate limited", 503: "server issue"}
assert status_examples[200] == "success" and status_examples[429] == "rate limited"
print("Status code examples:", status_examples)

In [ ]:
# Optional live demonstration. Leave False for a deterministic, offline session.
RUN_LIVE_DEMO = False
if RUN_LIVE_DEMO:
    live_payload = get_json(
        "https://openlibrary.org/search.json",
        params={"title": "The Hobbit", "limit": 3},
        timeout=10,
    )
    live_titles = [book.get("title", "(untitled)") for book in live_payload.get("docs", [])]
    assert isinstance(live_titles, list)
    print("Titles returned by Open Library:", live_titles)
else:
    print("Live API request skipped; all remaining examples use local sample data.")

## 3. JSON and nested records

JSON represents objects, arrays and scalar values. API responses often nest useful fields—for example, a title under `metadata` and an author's name under `author`. Inspect the shape first, then select or normalise the fields required for analysis. `pandas.json_normalize` flattens nested objects; it does not decide which fields are meaningful.

The generated raw CSV is a reusable, reproducible snapshot with provenance. Its schema is `record_id`, `title`, `author`, `category`, `price_gbp`, `available`, `source`, `source_url`, `collected_at_utc`. After flattening the JSON, only explicitly selected fields enter the analysis dataset.

In [ ]:
DATA_DIR = Path("data") / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_CSV = DATA_DIR / "catalogue_raw.csv"
JSON_PATH = DATA_DIR / "catalogue_response.json"
XML_PATH = DATA_DIR / "catalogue_response.xml"

books = [
    {"record_id": "B001", "title": "The Hobbit", "author": "J. R. R. Tolkien", "category": "Fantasy", "price_gbp": 8.99, "available": True},
    {"record_id": "B002", "title": "Pride and Prejudice", "author": "Jane Austen", "category": "Fiction", "price_gbp": 7.50, "available": True},
    {"record_id": "B003", "title": "A Brief History of Time", "author": "Stephen Hawking", "category": "Science", "price_gbp": 9.25, "available": False},
    {"record_id": "B004", "title": "The Left Hand of Darkness", "author": "Ursula K. Le Guin", "category": "Science fiction", "price_gbp": 8.25, "available": True},
]

# Save a reusable raw snapshot with provenance columns for later sessions.
raw_fields = [
    "record_id", "title", "author", "category", "price_gbp", "available",
    "source", "source_url", "collected_at_utc",
]
with RAW_CSV.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=raw_fields)
    writer.writeheader()
    for book in books:
        writer.writerow({
            **book,
            "source": "local sample",
            "source_url": "local://catalogue",
            "collected_at_utc": "2026-10-26T00:00:00Z",
        })

# Create nested JSON and XML response fixtures from the same deterministic records.
json_items = [
    {
        "record_id": book["record_id"],
        "metadata": {
            "title": book["title"],
            "category": book["category"],
            "price_gbp": book["price_gbp"],
            "available": book["available"],
        },
        "author": {"name": book["author"]},
    }
    for book in books
]
JSON_PATH.write_text(json.dumps({"items": json_items}, indent=2), encoding="utf-8")

root = ET.Element("catalogue")
for book in books:
    item = ET.SubElement(root, "book", record_id=book["record_id"])
    for field in ("title", "author", "category", "price_gbp", "available"):
        ET.SubElement(item, field).text = str(book[field])
ET.ElementTree(root).write(XML_PATH, encoding="utf-8", xml_declaration=True)

assert RAW_CSV.exists() and JSON_PATH.exists() and XML_PATH.exists()
print(f"Created {len(books)} deterministic records in {RAW_CSV}")
print("Raw CSV schema:", ", ".join(raw_fields))

In [ ]:
payload = json.loads(JSON_PATH.read_text(encoding="utf-8"))
assert list(payload) == ["items"] and len(payload["items"]) == 4

json_flat = pd.json_normalize(payload["items"], sep="_").rename(columns={
    "metadata_title": "title",
    "author_name": "author",
    "metadata_category": "category",
    "metadata_price_gbp": "price_gbp",
    "metadata_available": "available",
})
json_records = json_flat[["record_id", "title", "author", "category", "price_gbp", "available"]]
assert json_records.loc[0, "author"] == "J. R. R. Tolkien"
print(json_records.head(2).to_string(index=False))

## 4. CSV and XML: parse and compare

**CSV** is compact and convenient for tabular records, but has little built-in type or nesting information; values are commonly read as text and need validation or conversion. **XML** is structured and supports attributes and nested elements, but typically needs more parsing code. **JSON** maps naturally to Python dictionaries and lists and is common in REST APIs, though its nesting still needs deliberate normalisation.

No format is automatically better: use the source format you receive, then validate and convert it to the shape your project needs.

In this example CSV needed numeric and Boolean validation, while XML attributes and element text were explicitly converted. Assertions catch malformed values early.

In [ ]:
csv_source = pd.read_csv(RAW_CSV, dtype={"record_id": "string"})
csv_records = csv_source[["record_id", "title", "author", "category", "price_gbp", "available"]].copy()
csv_records["price_gbp"] = pd.to_numeric(csv_records["price_gbp"], errors="raise")
csv_records["available"] = csv_records["available"].map({True: True, False: False, "True": True, "False": False})
assert csv_records["price_gbp"].notna().all()
assert csv_records["available"].notna().all()

xml_root = ET.parse(XML_PATH).getroot()
xml_rows = []
for item in xml_root.findall("book"):
    xml_rows.append({
        "record_id": item.attrib["record_id"],
        "title": item.findtext("title"),
        "author": item.findtext("author"),
        "category": item.findtext("category"),
        "price_gbp": float(item.findtext("price_gbp")),
        "available": item.findtext("available") == "True",
    })
xml_records = pd.DataFrame(xml_rows)
assert len(csv_records) == len(xml_records) == 4
assert xml_records.loc[0, "price_gbp"] == 8.99
print("CSV and XML parsed successfully:", len(csv_records), "and", len(xml_records), "records")

## 5. Guided exercise — build one consistent DataFrame

Combine the records represented in JSON, CSV and XML. They are three encodings of the same catalogue, so use `record_id` to remove duplicates. Check the output shape, required columns, missing values, types and key uniqueness before using it downstream.

**Expected result:** four unique rows and six canonical columns; `price_gbp` is numeric, `available` is Boolean, and required values are present. Keep the richer raw CSV separately so later sessions can revisit the source snapshot and provenance.

In [ ]:
canonical_columns = ["record_id", "title", "author", "category", "price_gbp", "available"]
catalogue_df = pd.concat(
    [json_records[canonical_columns], csv_records[canonical_columns], xml_records[canonical_columns]],
    ignore_index=True,
).drop_duplicates(subset="record_id", keep="first")
catalogue_df["price_gbp"] = pd.to_numeric(catalogue_df["price_gbp"], errors="raise")
catalogue_df["available"] = catalogue_df["available"].astype(bool)

assert catalogue_df.shape == (4, 6)
assert catalogue_df.columns.tolist() == canonical_columns
assert catalogue_df["record_id"].is_unique
assert not catalogue_df[canonical_columns].isna().any().any()
assert (catalogue_df["price_gbp"] >= 0).all()
print(catalogue_df.to_string(index=False))
print("Validated shape:", catalogue_df.shape)

## 6. Independent task — 15 minutes

Create a small collection workflow from `catalogue_df` that:
1. selects available books;
2. validates required columns, unique record IDs and non-negative numeric prices;
3. returns only the canonical fields, sorted by title.

Try implementing it before running the reference cell. Extend it with one useful quality check of your own. The checks should fail loudly if the input is invalid.

In [ ]:
def collect_available_books(records):
    required = {"record_id", "title", "author", "category", "price_gbp", "available"}
    missing = required.difference(records.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = records.copy()
    result["price_gbp"] = pd.to_numeric(result["price_gbp"], errors="raise")
    if result["record_id"].isna().any() or not result["record_id"].is_unique:
        raise ValueError("record_id values must be present and unique")
    if result["price_gbp"].isna().any() or (result["price_gbp"] < 0).any():
        raise ValueError("price_gbp values must be present and non-negative")
    if result["available"].isna().any():
        raise ValueError("available values must be present")

    return result.loc[result["available"], canonical_columns].sort_values("title").reset_index(drop=True)

available_books = collect_available_books(catalogue_df)
assert len(available_books) == 3
assert available_books["title"].is_monotonic_increasing
assert available_books["record_id"].is_unique
print(available_books[["record_id", "title", "price_gbp"]].to_string(index=False))
print("Validated available records:", len(available_books))

## 7. Reflect and recap

Discuss or write a short answer to each question:
1. Which source would you choose for a catalogue that must be refreshed every hour, and why?
2. What should your client do when an API returns `429` or `503`?
3. Why should a request have a timeout, and why should HTTP status be checked before JSON parsing?
4. What did normalisation change about the nested JSON, and what information might you choose not to keep?
5. Which format would you prefer for a flat tabular hand-off, and which for a nested API response? What trade-offs matter?

**Expected points:** (1) a documented API if its terms, fields and rate limits fit the refresh requirement; (2) respect rate limits and retry transient server failures cautiously, with back-off; (3) avoid hanging and distinguish transport/HTTP failures from invalid response bodies; (4) nested values became selected flat columns, while irrelevant or sensitive fields can be excluded; (5) CSV is simple for flat tables, JSON is natural for nested API data, and the best choice depends on the receiving workflow.

**Recap:** choose a permitted source deliberately; request data with parameters, timeouts and explicit error checks; parse the source format; normalise, validate and persist records. The reusable raw file is `data/raw/catalogue_raw.csv`.

**Raw dataset schema for later sessions:** `record_id` (unique string), `title` (string), `author` (string), `category` (string), `price_gbp` (non-negative numeric GBP), `available` (Boolean), `source` (source label), `source_url` (source locator), `collected_at_utc` (ISO-8601 UTC timestamp).